# 02 - Neural networks across three evaluation protocols

Research question: does the choice of evaluation protocol affect reported NIDS
performance more than the choice of neural architecture?

Two factors, three levels each:

| | P1 naive | P2 corrected | P3 zero-day |
|---|---|---|---|
| A1 MLP | | | |
| A2 1D-CNN | | | |
| A3 Autoencoder | | | |

Read rows for the architecture effect, columns for the protocol effect, then
compare the two spreads. That comparison is the answer.

Gold tables come from notebook 01. The protocol is a property of the data, so
all nine cells train identical code against identical interfaces.

## 0. Setup

In [ ]:
import json, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import (precision_recall_fscore_support, confusion_matrix,
                             average_precision_score, matthews_corrcoef)

warnings.filterwarnings("ignore")

PROJECT = Path(r"D:\Saad A2")
GOLD    = PROJECT / "lakehouse" / "gold"
RESULTS = PROJECT / "results"
RESULTS.mkdir(exist_ok=True)

BENIGN   = "Benign"
ZERO_DAY = "Bot"
PROTOCOLS = ["p1_naive", "p2_corrected", "p3_zeroday"]

# classes present in all three partitions (written by notebook 01 cell 7.3b)
_ec = json.loads((RESULTS / "eval_classes.json").read_text())
EVAL_CLASSES = _ec["eval_classes"]
print(f"{len(EVAL_CLASSES)} classes scored, {len(_ec['excluded'])} excluded")

# how many seeds per cell. 3 gives a variance estimate, 1 is a quick pass.
N_SEEDS   = 3
MAX_EPOCH = 25
PATIENCE  = 4
BATCH     = 4096   # large batch: far fewer optimiser steps on CPU

torch.set_num_threads(4)
DEVICE = "cpu"
print(f"torch {torch.__version__} on {DEVICE}, {torch.get_num_threads()} threads")

15 classes scored, 0 excluded
torch 2.13.0+cpu on cpu, 4 threads


## 1. Loading and preprocessing

Scaler and imputation values are fitted on train only. Fitting them on the full
dataset would leak test statistics into training - the same class of mistake
this study is about.

In [ ]:
def load_split(protocol, split):
    """Read one Gold partition as float32 features plus string labels."""
    df = pd.read_parquet(GOLD / protocol / split)
    y = df["Label"].to_numpy()
    X = df.drop(columns=["Label"]).astype("float32")
    return X, y


def prepare(protocol):
    """Load all three splits, impute and scale using train statistics only."""
    Xtr, ytr = load_split(protocol, "train")
    Xva, yva = load_split(protocol, "val")
    Xte, yte = load_split(protocol, "test")
    cols = list(Xtr.columns)

    med = Xtr.median()                      # train only
    Xtr, Xva, Xte = (d.fillna(med) for d in (Xtr, Xva, Xte))

    mu, sd = Xtr.mean(), Xtr.std().replace(0, 1.0)      # train only
    Xtr, Xva, Xte = ((d - mu) / sd for d in (Xtr, Xva, Xte))

    out = []
    for d in (Xtr, Xva, Xte):
        a = d.to_numpy(dtype="float32")
        a = np.clip(np.nan_to_num(a, nan=0.0, posinf=0.0, neginf=0.0), -10, 10)
        out.append(a)
    return dict(Xtr=out[0], ytr=ytr, Xva=out[1], yva=yva,
                Xte=out[2], yte=yte, cols=cols)


DATA = {}
for p in PROTOCOLS:
    t0 = time.perf_counter()
    DATA[p] = prepare(p)
    d = DATA[p]
    print(f"{p:14} train {d['Xtr'].shape}  val {d['Xva'].shape}  "
          f"test {d['Xte'].shape}  ({time.perf_counter()-t0:.1f}s)")
print(f"\nP1 has {DATA['p1_naive']['Xtr'].shape[1] - DATA['p2_corrected']['Xtr'].shape[1]} "
      f"extra features - the leaky ones kept for the naive protocol")

p1_naive       train (570003, 79)  val (121860, 79)  test (121785, 79)  (7.0s)
p2_corrected   train (528602, 77)  val (114133, 77)  test (111968, 77)  (5.2s)
p3_zeroday     train (500654, 77)  val (108145, 77)  test (111968, 77)  (5.8s)

P1 has 2 extra features - the leaky ones kept for the naive protocol


In [ ]:
# binary target: benign vs attack. All three architectures can produce this,
# so it is the common ground for comparing the grid. Multi-class macro-F1 is
# reported separately for the two supervised models.
def to_binary(y):
    return (y != BENIGN).astype("int64")

for p in PROTOCOLS:
    d = DATA[p]
    for s in ("tr", "va", "te"):
        d[f"b{s}"] = to_binary(d[f"y{s}"])
    print(f"{p:14} attack rate  train {d['btr'].mean():.3f}  "
          f"val {d['bva'].mean():.3f}  test {d['bte'].mean():.3f}")

p1_naive       attack rate  train 0.508  val 0.509  test 0.509
p2_corrected   attack rate  train 0.470  val 0.473  test 0.466
p3_zeroday     attack rate  train 0.441  val 0.444  test 0.466


## 2. Architectures

Choice is driven by the data, not by what is fashionable:

- records are tabular, 77-79 numeric features, no structure within a record
- each flow is labelled independently, so row order is not a sequence
- heavy class imbalance
- one architecture must be able to flag an attack class it never saw

No LSTM or transformer. Each row is already a time-aggregated summary of a flow,
so feeding consecutive rows to a sequence model would treat file order as time,
which it is not - adjacent rows are unrelated flows between different hosts.

In [ ]:
class MLP(nn.Module):
    """A1. Dense net over the feature vector. The natural fit for tabular data."""
    def __init__(self, n_in, n_out=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, 128), nn.ReLU(), nn.BatchNorm1d(128), nn.Dropout(0.3),
            nn.Linear(128, 64),   nn.ReLU(), nn.BatchNorm1d(64),  nn.Dropout(0.2),
            nn.Linear(64, n_out))
    def forward(self, x):
        return self.net(x)


class CNN1D(nn.Module):
    """A2. Treats the feature vector as a 1D signal.

    Caveat worth stating in the paper: column order in a flow record is
    arbitrary, and convolution assumes adjacent positions are related. If this
    beats the MLP the gain may be an artefact of column ordering, which a
    column-shuffle ablation would settle.
    """
    def __init__(self, n_in, n_out=2):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 16, 5, padding=2),  nn.ReLU(), nn.BatchNorm1d(16), nn.MaxPool1d(2),
            nn.Conv1d(16, 32, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(32), nn.MaxPool1d(2))
        self.head = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.3),
            nn.Linear(32 * (n_in // 4), 64), nn.ReLU(),
            nn.Linear(64, n_out))
    def forward(self, x):
        return self.head(self.conv(x.unsqueeze(1)))


class AutoEncoder(nn.Module):
    """A3. Trained on benign traffic only; anomaly score is reconstruction error.

    The only architecture here that can structurally detect an unseen attack
    class, because it never consumes attack labels (Mirsky et al. 2018).
    """
    def __init__(self, n_in):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(n_in, 64), nn.ReLU(),
                                 nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 16))
        self.dec = nn.Sequential(nn.Linear(16, 32), nn.ReLU(),
                                 nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, n_in))
    def forward(self, x):
        return self.dec(self.enc(x))